# Model Interpretation (Phase 6-7)

> Goal: translate evaluation outputs into business decisions for retention strategy.

This notebook focuses on interpretation of exported metrics and strategy artifacts, not on retraining models.

In [ ]:
from pathlib import Path
import json
import ast

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent
metrics_dir = PROJECT_ROOT / "reports" / "metrics"
figures_eval_dir = PROJECT_ROOT / "reports" / "figures" / "evaluation"

evaluation_df = pd.read_csv(metrics_dir / "evaluation_summary.csv")
strategy = json.loads((metrics_dir / "model_decision_strategy.json").read_text(encoding="utf-8"))

sns.set_theme(style="whitegrid")
evaluation_df

## 1) Policy Snapshot

This is the production decision policy exported by training.

In [ ]:
policy_table = pd.json_normalize(strategy, sep=".").T.rename(columns={0: "value"})
policy_table

In [ ]:
plot_df = evaluation_df[["model_label", "strategy_label", "precision", "recall", "f1_score", "roc_auc"]].copy()
plot_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=plot_df.sort_values("recall", ascending=False),
    x="strategy_label",
    y="recall",
    hue="model_label",
    ax=axes[0],
)
axes[0].set_title("Recall by Strategy")
axes[0].set_xlabel("Strategy")
axes[0].set_ylabel("Recall")

sns.barplot(
    data=plot_df.sort_values("precision", ascending=False),
    x="strategy_label",
    y="precision",
    hue="model_label",
    ax=axes[1],
)
axes[1].set_title("Precision by Strategy")
axes[1].set_xlabel("Strategy")
axes[1].set_ylabel("Precision")

for ax in axes:
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 2) Confusion-Matrix Interpretation

Below we convert stored confusion matrices into rates that are easier to discuss with business stakeholders.

In [ ]:
cm_rows = []
for _, row in evaluation_df.iterrows():
    matrix = ast.literal_eval(row["confusion_matrix"])
    tn, fp = matrix[0]
    fn, tp = matrix[1]
    total = tn + fp + fn + tp

    cm_rows.append(
        {
            "model_label": row["model_label"],
            "strategy_label": row["strategy_label"],
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp,
            "true_positive_rate_recall": tp / (tp + fn),
            "false_positive_rate": fp / (fp + tn),
            "contact_rate_predicted_positive": (tp + fp) / total,
        }
    )

cm_df = pd.DataFrame(cm_rows)
cm_df

In [ ]:
gb_default = cm_df.query("model_label == 'gradient_boosting' and strategy_label == 'default_threshold'").iloc[0]
gb_policy = cm_df.query("model_label == 'gradient_boosting' and strategy_label == 'policy_threshold'").iloc[0]

delta_recall = gb_policy["true_positive_rate_recall"] - gb_default["true_positive_rate_recall"]
delta_contact = gb_policy["contact_rate_predicted_positive"] - gb_default["contact_rate_predicted_positive"]

pd.DataFrame(
    {
        "metric": ["recall_delta", "contact_rate_delta"],
        "value": [delta_recall, delta_contact],
    }
)

## 3) Recommendation

- Use Gradient Boosting + default threshold for precision-sensitive outreach.
- Use Gradient Boosting + policy threshold for recall-first retention drives.
- Use SVM as a challenger in experiments where broader targeting is acceptable.

In all cases, monitor campaign cost per contacted customer alongside churn capture rate.